## This is used for store the embedding data into database

FAISS: Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [8]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

In [11]:
#Load the text or infromation

loader=TextLoader("speech.txt")

docs=loader.load()
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness be

In [14]:
#Chunk tthe document for Model effeciency
chunk_document=CharacterTextSplitter( chunk_size=200,chunk_overlap=20)

chunk_document


In [16]:
final_document=chunk_document.split_documents(docs)

Created a chunk of size 470, which is longer than the specified 200
Created a chunk of size 347, which is longer than the specified 200
Created a chunk of size 668, which is longer than the specified 200
Created a chunk of size 982, which is longer than the specified 200
Created a chunk of size 789, which is longer than the specified 200


In [17]:
final_document

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'),
 Document(metadata={'source': 'speech.txt'}, page_content

In [18]:
##Embedding the data and store it into the FAISS datastore

embedding=OllamaEmbeddings(model="nomic-embed-text")
embedding

OllamaEmbeddings(model='nomic-embed-text', dimensions=None, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [19]:
db=FAISS.from_documents(final_document,embedding)

In [23]:
#Query infromation from FAISS

query="How does the speaker describe the desired outcome of the war?"

docs=db.similarity_search(query)
docs[0].page_content

'It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers.Using FAISS as a retriever means you are using it as the "search engine" or "lookup boy" for a Retrieval-Augmented Generation (RAG) pipeline or an information retrieval system.
Instead of asking an AI model to answer a question purely from its memory, the retriever intercepts the question, searches a database of your documents to find the most relevant paragraphs, and hands those paragraphs to the AI to read before it answers. 

In [26]:
retriever=db.as_retriever()
docs=retriever.invoke(query)
docs[0].page_content

'It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [29]:
docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='6d5d036d-16bf-495a-b6ae-3e7fe48a6d4a', metadata={'source': 'speech.txt'}, page_content='It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
  np.float32(0.9204212)),
 (Document(id='111262ff-dcd3-4f83-81d6-7225987f1047', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in th

In [32]:
embedding_vector=embedding.embed_query(query)
embedding_vector

[-0.017979331,
 0.006356286,
 -0.15973534,
 -0.045196533,
 0.10099887,
 0.07276679,
 -0.045548003,
 0.006372562,
 0.01745092,
 -0.0009231483,
 -0.008624126,
 0.02970444,
 0.04434047,
 0.06834731,
 0.094856605,
 -0.04401382,
 0.014197934,
 -0.045699086,
 -0.035662543,
 0.021447023,
 -0.02955745,
 -0.012397554,
 6.307233e-05,
 0.031192293,
 0.071599185,
 0.024628423,
 -0.020611718,
 0.039738387,
 -0.03853724,
 -0.03616693,
 0.04562593,
 0.0048233583,
 -0.0064330697,
 0.004354834,
 -0.07234146,
 -0.07703081,
 0.0036928714,
 0.06226035,
 0.013746594,
 -0.059902664,
 -0.006032113,
 -0.026603898,
 0.0018848914,
 -0.07706441,
 0.05504515,
 -0.020084014,
 -0.018463153,
 0.0059770453,
 -4.565709e-05,
 -0.027827231,
 -0.03570902,
 -0.029578902,
 0.004856763,
 -0.010857679,
 0.057140436,
 0.0063062673,
 -0.033553027,
 -0.03464481,
 0.031641558,
 0.0029145465,
 0.024647595,
 0.010240354,
 -0.02693362,
 0.03752061,
 0.05970913,
 -0.031599022,
 -0.06040069,
 0.029940713,
 0.025159204,
 -0.05912426,


## Saves those file and Load

In [33]:
db.save_local("faiss_index")

In [34]:
##Load the file document from faiss_index and pkl file

new_db=FAISS.load_local("faiss_index",embedding,allow_dangerous_deserialization=True)

In [35]:
docs=new_db.similarity_search(query)

In [36]:
docs

[Document(id='6d5d036d-16bf-495a-b6ae-3e7fe48a6d4a', metadata={'source': 'speech.txt'}, page_content='It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='111262ff-dcd3-4f83-81d6-7225987f1047', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are